# Exploring a Dataset


Every spatial analysis begins by looking at the data. You need to know the structure of the dataset, what is in it, and its key properties:

- what types of features the dataset contains;
- what geometry type is used (points, lines, polygons);
- what coordinate reference system the data is in;
- what attributes are available for analysis;
- whether there are missing values, duplicates, or potential errors;
- and, before any of that, what the data looks like on a map.

> In this section, we will look at how to carry out an initial exploration of a spatial dataset in **GeoDataFrame** format.


## 0. Importing Libraries and Preparing the Data


Import libraries


In [ ]:
import geopandas as gpd
import pandas as pd

- [**GeoPandas**](https://geopandas.org/) (`geopandas`) — a Python library that extends pandas to support geospatial data. With it you can load, process and analyse spatial datasets in a range of formats.

- [**pandas**](https://pandas.pydata.org/) (`pandas`) — used here to read one plain CSV file, for a comparison further down.


Prepare the data


All examples in this section use **vienna_metro.geojson** from `data/vienna/` — the U-Bahn stations of Vienna. _Every dataset and its source is listed on the [Course Modules](../module_0/syllabus.md) page._

You are welcome to use your own data instead (this is even encouraged).

Start by reading the `.geojson` file.


In [ ]:
gdf = gpd.read_file("../../data/vienna/vienna_metro.geojson")

## 1. Basic Information


### 1.1. Previewing the First Rows

The `head()` method displays the first rows of a `DataFrame` or `GeoDataFrame`.
It is used to get an initial look at the data.

By default, it returns the first **5 rows**.


In [ ]:
gdf.head()

You can also specify the number of rows manually:


In [ ]:
gdf.head(3)

### 1.2. General Information

The `info()` method displays a concise summary of a `DataFrame` or `GeoDataFrame`:

- the number of records (rows),
- column names,
- the data type of each column,
- the count of **non-null values**,
- the memory usage of the dataset.


In [ ]:
gdf.info()

The `describe()` method complements `info()`: where `info()` reports the **structure**, `describe()` summarises the **values** of the numeric columns — count, mean, standard deviation, minimum, maximum and the quartiles.

In [ ]:
gdf.describe()

Read the result critically. `describe()` will compute statistics for any numeric column, whether or not they mean anything: `opened_year` gives a genuine picture of when the network was built, while the mean of `line` — a label that merely happens to be written as a number — means nothing at all.

### 1.3. Size

To understand the size of a dataset, the `len()` function and the `shape` attribute are particularly useful.

`len()` returns the number of rows, i.e. the **number of features**.


In [ ]:
len(gdf)

The `shape` attribute returns a tuple of two numbers:

- the first number is the number of rows (features),
- the second is the number of columns (attributes, including `geometry`).


In [ ]:
gdf.shape

### 1.4. A First Look at the Map

Everything so far describes the table. For spatial data the quickest check is to put the features on a map, because it exposes at a glance what no summary will report: coordinates entered in the wrong order, a layer stranded off the coast of Africa because the CRS is wrong, one stray feature on another continent.

`explore()` builds an interactive map straight from the `GeoDataFrame`.

In [ ]:
gdf.explore(tiles="cartodbpositron")

The stations sit where Vienna is, in the shape of the U-Bahn network. Had any of the problems above been present, this single cell would have shown it immediately.

### 1.5. Spatial Extent

The map answers "where is this data" by eye; `total_bounds` answers it in numbers. The **bounding box** is the smallest rectangle that fully encloses all features, given as four coordinates:

```
[minx, miny, maxx, maxy]
```

Read against a place you know, it is a fast sanity check: Vienna sits at roughly 16.2–16.6 degrees east and 48.1–48.3 north, so anything far from that would mean the coordinates, or the CRS, are wrong.


In [ ]:
gdf.total_bounds

The numbers match: the whole layer sits inside Vienna.


## 2. Geometry


In a `GeoDataFrame`, every feature has a **geometry**.

Before you analyse anything, answer a few questions:

- what geometry type is used (points, lines, polygons);
- whether any geometry values are missing;
- whether all geometries are valid.


### 2.1. Geometry Column

In a `GeoDataFrame`, spatial information is stored in the special `geometry` column.
This is what distinguishes a `GeoDataFrame` from a standard pandas `DataFrame`.

Accessing this attribute returns a `GeoSeries` containing the geometry of each spatial feature.

The first five values:


In [ ]:
gdf.geometry[:5]

#### 2.1.1. Missing Values

Datasets sometimes contain features with no geometry, which can cause spatial analysis tools to fail.
It is therefore important to check for missing geometries upfront.

- `isna()` — identifies missing values;
- `sum()` — counts how many there are.


In [ ]:
gdf.geometry.isna().sum()

If the result is greater than zero, some rows contain `NaN` geometry and need to be handled (e.g. removed or repaired).

Rows with missing geometry can be removed as follows:


In [ ]:
gdf = gdf[gdf.geometry.notna()]

#### 2.1.2. Empty Geometries

In addition to missing values (`NaN`), a dataset may contain **empty geometries** (`EMPTY`).
These features technically have a geometry object, but it carries no spatial information.

The check follows the same pattern as above, with `is_empty` in place of `isna()`.


In [ ]:
gdf.geometry.is_empty.sum()

If the result is greater than zero, the dataset contains empty geometries that need to be removed or repaired.

Rows with empty geometry can be removed as follows:


In [ ]:
gdf = gdf[~gdf.geometry.is_empty]

#### 2.1.3. Active Geometry Column

A `GeoDataFrame` always has an **active geometry column** — the one used for all spatial operations.

By default, this column is named `geometry`, but the name may differ (for example, after renaming the column or creating a new geometry).

The `geometry.name` attribute returns the name of the currently active geometry column.


In [ ]:
gdf.geometry.name

### 2.2. Geometry Type

The `geom_type` attribute is used to determine the geometry type of each feature.
It returns a `GeoSeries` with the geometry type listed for each row.

The first five:


In [ ]:
gdf.geom_type[:5]

A dataset may sometimes contain features with mixed geometry types.
Which types are present, and how many features of each:


In [ ]:
gdf.geom_type.value_counts()

Our dataset is points and nothing else, so the count has a single row. That is the ordinary case, but not the only one. Here is the same check on the land use layer from the [sixth module](../module_6/map_1.ipynb):

In [ ]:
landuse = gpd.read_file("../../data/leopoldstadt/landuse.geojson")

landuse.geom_type.value_counts()

Most areas are a single `Polygon`, but twelve are a `MultiPolygon` — one land use area made of several disconnected pieces, a park split by a road, for example.

Mixed types are worth spotting early, because a layer is not always what you expect it to be: OpenStreetMap stores some cafés as building outlines rather than as points, so a query for cafés can return polygons alongside the points. Code written for points alone will then fail, or worse, quietly skip them.

### 2.3. Geometry Validity

Beyond missing and empty geometries, features may also have **invalid geometry**.

Validity issues are most common in polygon datasets, for example:

- self-intersections;
- unclosed rings;
- topology errors.

The `is_valid` attribute checks the validity of each feature's geometry, returning `True` or `False` for each one.
Count the invalid geometries in the dataset.

Our dataset consists of points, which are always valid, so the result here will be `0`. For polygon datasets this check is essential.


In [ ]:
(~gdf.geometry.is_valid).sum()

You can also inspect which features specifically have invalid geometry:


In [ ]:
gdf[~gdf.geometry.is_valid]

Invalid geometry can be repaired as follows:


In [ ]:
gdf = gdf.set_geometry(gdf.geometry.make_valid())

After repairing, it is worth running the check again to confirm that no invalid geometries remain:


In [ ]:
(~gdf.geometry.is_valid).sum()

## 3. Coordinate Reference System (CRS)

The geometries in a `GeoDataFrame` are defined in a specific **Coordinate Reference System**
(**CRS**).

Before you analyse anything, answer a few questions:

- whether a CRS is defined for the dataset;
- what type of CRS it is;
- whether the current CRS is suitable for calculating distances and areas;
- whether all layers being used share the same CRS (comparing and transforming coordinate systems is covered in the second module).

In this section, we will look at the key CRS properties that can be checked during initial data exploration.
Coordinate reference systems will be covered in greater depth in the second module.


### 3.1. General Information — `.crs`

The `crs` attribute returns the coordinate reference system of the `GeoDataFrame`.

If no CRS is defined, the attribute will return `None`.


In [ ]:
gdf.crs

From this output we can see the following properties of our dataset's CRS:

- It is a geographic coordinate system with the identifier EPSG:4326.
- Name: WGS 84.
- Axes: latitude (Lat) and longitude (Lon), measured in degrees.
- Area of use: the entire world.
- Ellipsoid: WGS 84.
- Prime meridian: Greenwich.

Note the axis order reported here: the CRS definition lists latitude first and longitude second. In shapely and geopandas, however, coordinates are always stored as `(x, y)` — longitude first, latitude second — as in the `Point(30.2963, 59.9255)` example from the first section.


### 3.2. CRS Type

A coordinate reference system can be either:

- **geographic** — coordinates expressed in degrees (latitude and longitude);
- **projected** — coordinates expressed in linear units (most commonly metres).

The CRS type can be determined using the `.is_geographic` and `.is_projected` attributes.


In [ ]:
print(f"Geographic: {gdf.crs.is_geographic}")
print(f"Projected: {gdf.crs.is_projected}")

### 3.3. EPSG Code

Sometimes it is convenient to retrieve just the **numeric EPSG code** of the coordinate reference system.

This can be done using the `crs.to_epsg()` method.


In [ ]:
gdf.crs.to_epsg()

If the CRS is non-standard, the method may return `None`.


## 4. Attributes

Besides geometry, every feature in a `GeoDataFrame` carries **attributes** — the descriptive properties that come with it.

In this section, we will look at how to examine the structure of attribute data, check column types, and identify missing values.


### 4.1. List of Columns

The `columns` attribute returns a list of all columns in a `DataFrame` or `GeoDataFrame`.
It returns a pandas `Index` object containing the names of all columns, including the geometry column.


In [ ]:
gdf.columns

### 4.2. Data Types

The `dtypes` attribute returns the data type of each column in a `DataFrame` or `GeoDataFrame`.


In [ ]:
gdf.dtypes

Checking data types ensures that each column is in the correct format and that any necessary transformations can be performed without errors.


### 4.3. Missing Attribute Values

To judge the quality of the attribute data, check how many values are missing in each column.


In [ ]:
gdf.isna().sum()

Every column of this file is filled in, which is a comfortable but unusual position to be in. A more typical result looks like the one below — the same check on the tourist locations file, where the contact details are patchy:

In [ ]:
locations = pd.read_csv("../../data/vienna/vienna_top_locations.csv", sep=";", decimal=",")

locations.isna().sum()

Eighty of the 135 locations have no email address. Whether that matters depends entirely on the task: if you are putting the locations on a map, the gap is irrelevant; if you meant to contact them, it is the main finding of your exploration.

### 4.4. Counting the Values

For a column holding a limited set of repeating values, `value_counts()` shows how the features are distributed between them. It is the most useful single check you can run on an attribute.

In [ ]:
gdf["line"].value_counts().sort_index()

The stations are spread fairly evenly across the five lines, which is what we would expect.

Run the same method on a category, a district or a status column and it answers two questions at once: whether the data covers what you assumed it did, and whether some values are misspelled variants of others — `musuem` alongside `museum` in the locations file above, for instance.

## 5. Duplicates


The dataset may contain fully duplicated rows, or different features sharing exactly the same geometry (a feature digitised twice, for example).

`duplicated()` flags the repeats, counted the same way.


In [ ]:
# fully duplicated rows
print(f"Duplicate rows: {gdf.duplicated().sum()}")

# features sharing the same geometry
print(f"Duplicate geometries: {gdf.geometry.duplicated().sum()}")

Duplicate geometries are not always an error: two records can legitimately describe the same place — the separate entrances of one station, say — or the same feature may have been digitised twice by mistake. In this dataset both counts come back zero, which is what you want to see. Had they not, whether to merge, remove or keep the duplicates would depend on the task at hand.


## Summary


In this section, we covered the main approaches to initial exploration of a spatial dataset in `GeoDataFrame` format: reading the table with `head()`, `info()` and `describe()`, putting it on a map, checking the geometry, the coordinate reference system and the attributes, and looking for duplicates.

This is a crucial step when working with spatial datasets: it helps you understand the structure and contents of the data, verify that it has been loaded correctly, and identify any potential errors.

Two habits are worth carrying forward. **Look at the map first** — it catches in one glance the kind of error that no table summary reports. And **do not read a clean result as proof that the data is clean**: our metro file passed every check without a single problem, which is not the usual state of affairs, as the two comparisons above showed.
